# Text Generation

In this lab, we are going to simply experiment with a few text generation models to discover associated APIs.

## Importing the dependencies

First, we are going to import all the dependencies that we will need for this lab. If you cannot run the following code cell, do not forget to [create an environment](https://docs.astral.sh/uv/guides/projects/), to install the dependencies inside of it (using the command `uv add -r requirements.txt`) and to use it as your Jupyter kernel.

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['HF_HOME'] = os.getcwd() + "/cache/"

import torch
from transformers import set_seed, AutoTokenizer, AutoModelForCausalLM

## Identifying the best device to run the model

Since we are going to perform a computing-intensive task, we must identify the most efficient device available to perform it. We do so using PyTorch, which is the back-end that we will use in this lab. We prioritize NVIDIA GPUs with CUDA installed, then Apple Silicon GPUs, and finally CPUs if none of the above is found.

If you need help installing the relevant version of PyTorch: https://pytorch.org/get-started/locally/

If you have a NVIDIA GPU but you don't know whether you have CUDA installed or not, type the following command:

```bash
nvcc --version
```

If you have it installed, you should see the CUDA version installed on your computer. Otherwise, you should install a PyTorch-compatible version (as listed [here](https://pytorch.org/get-started/locally/), row "Stable CUDA").

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device('cpu')

print(device)

## Text completion

First, let's download and prepare the language model that we will use to do some text completion! This model is GPT2, a fairly small language model made for text generation.

In [ ]:
model_name = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
completion_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

Now let's play with our model! First, we define our initial text, then we will see how we can complete it using the different strategies mentioned in the class.

### With hill-climbing search

In [ ]:
inputs = tokenizer("Today is the last class before the exam", return_tensors="pt").to(device)

outputs = completion_model.generate(**inputs, max_new_tokens=100, pad_token_id=tokenizer.eos_token_id)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

### With beam search (beam width = 10)

In [ ]:
inputs = tokenizer("Today is the last class before the exam", return_tensors="pt").to(device)

outputs = completion_model.generate(**inputs, max_new_tokens=100, pad_token_id=tokenizer.eos_token_id, num_beams=10, early_stopping=True)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

### Re-running the same methods with a hard-coded no repetition parameter

In [ ]:
inputs = tokenizer("Today is the last class before the exam", return_tensors="pt").to(device)

print("Using hill-climbing search and no 2-tokens repetition allowed:")
outputs = completion_model.generate(**inputs, max_new_tokens=100, no_repeat_ngram_size=2, pad_token_id=tokenizer.eos_token_id)
generated_text_hill = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text_hill)

print("\n" + "-" * 50 + "\n")

print("Using beam search and no 2-tokens repetition allowed:")
outputs = completion_model.generate(**inputs, max_new_tokens=100, no_repeat_ngram_size=2, pad_token_id=tokenizer.eos_token_id, num_beams=10, early_stopping=True)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

### With top-k sampling

In [ ]:
temperature = 1
k = 10

set_seed(12345)

inputs = tokenizer("Today is the last class before the exam", return_tensors="pt").to(device)

outputs = completion_model.generate(**inputs, max_new_tokens=100, pad_token_id=tokenizer.eos_token_id, do_sample=True, temperature=temperature, top_k=k)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)